### Crear embeddings

In [1]:
# -*- coding: utf-8 -*-
import os
import torch
import numpy as np
import pandas as pd
import sys
import json
from types import SimpleNamespace

# Ajusta esto si es necesario para importar tu modelo TuckER
sys.path.append('../..')
from model import TuckER

# ============================================================
# 🔹 CONFIGURACIÓN (2 COHORTES)
# ============================================================

# Carpeta del Grafo (Dataset 2019+2020)
# Ajusta esta ruta a donde tengas tus tripletas combinadas
data_dir = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"

# Carpeta de Salida
output_dir = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_2cohortes"
os.makedirs(output_dir, exist_ok=True)

# Archivos con notas (Historia: 2019-1 y 2020-1)
base_path = r"C:/Users/56946/TuckER/mis_scripts/dataframes_por_semestre"
csv_20191 = os.path.join(base_path, "df_20191.csv")
csv_20201 = os.path.join(base_path, "df_20201.csv")

# Archivo de Puntajes
path_puntajes = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Cursos fundamentales (4 dimensiones)
cursos_1er_semestre = ["MA1101", "MA1001", "FI1000", "BT1211"]

# Dimensiones del modelo (5 para incluir el puntaje)
edim, rdim = 5, 10
kwargs = {"input_dropout": 0.2, "hidden_dropout1": 0.2, "hidden_dropout2": 0.3}

# ============================================================
# 🔹 FUNCIONES AUXILIARES
# ============================================================

def leer_y_normalizar(path):
    if not os.path.exists(path):
        print(f"⚠️ Archivo no encontrado: {path}")
        return pd.DataFrame(columns=["ID", "CURSO", "NOTA"])
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip().str.upper()
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    relations = set()
    nombres = ["train.txt", "valid.txt", "test.txt", "train_balanceado.txt"]
    
    for file_name in nombres:
        path = os.path.join(data_dir, file_name)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 3:
                    h, r, t = parts[:3]
                    entities.add(h); entities.add(t); relations.add(r)
    
    relations_with_reverse = sorted(list(relations)) + [r + "_reverse" for r in sorted(list(relations))]
    return sorted(list(entities)), sorted(list(relations_with_reverse))

def heads_desde_tripletas(data_dir):
    heads = set()
    nombres = ["train.txt", "valid.txt", "test.txt"]
    for fname in nombres:
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1:
                    heads.add(parts[0])
    return heads

# ============================================================
# 🔹 1. CARGAR Y NORMALIZAR NOTAS (Rango -1 a 1)
# ============================================================

print("Cargando notas académicas (2019-1 y 2020-1)...")
df_19 = leer_y_normalizar(csv_20191)
df_20 = leer_y_normalizar(csv_20201)

# Concatenar ambas generaciones
df_total = pd.concat([df_19, df_20], ignore_index=True)

# Identificar universo de alumnos con notas S1
ids_con_historia = set(df_total["ID"].unique())
print(f"   Total alumnos con notas S1: {len(ids_con_historia)}")

df_total['NOTA'] = pd.to_numeric(df_total['NOTA'], errors='coerce')

# Normalización Notas: 1.0 -> -1.0, 7.0 -> 1.0 ((n-4)/3)
def escalar_nota(n):
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

df_total['NOTA'] = df_total['NOTA'].apply(escalar_nota)

def obtener_notas(df, cursos):
    # Pivot: ID x Cursos
    pivot = df[df["CURSO"].isin(cursos)].pivot_table(
        index="ID", columns="CURSO", values="NOTA", aggfunc="first"
    )
    return pivot.reindex(columns=cursos).fillna(-1.0)

notas_s1_matrix = obtener_notas(df_total, cursos_1er_semestre)

# ============================================================
# 🔹 2. CARGAR Y NORMALIZAR PUNTAJES (Rango -1 a 1)
# ============================================================

print("Cargando y normalizando puntajes...")
if os.path.exists(path_puntajes):
    df_puntajes = pd.read_csv(path_puntajes, sep=";")
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Calcular min/max usando los alumnos presentes en nuestras notas S1 (2019+2020)
    scores_gen = df_puntajes[df_puntajes["ID"].isin(ids_con_historia)]["PUNTAJE_PONDERADO"].dropna()
    
    min_score = scores_gen.min() if not scores_gen.empty else 450.0
    max_score = scores_gen.max() if not scores_gen.empty else 850.0
    
    print(f"   Rango Puntaje detectado (2019+2020): [{min_score}, {max_score}]")
    
    mapa_puntajes = {}
    
    # Función MinMax a [-1, 1]
    def minmax_scale(val, min_v, max_v):
        if max_v == min_v: return 0.0
        return 2 * (val - min_v) / (max_v - min_v) - 1

    for _, row in df_puntajes.iterrows():
        uid = row["ID"]
        score = row["PUNTAJE_PONDERADO"]
        
        if pd.isna(score):
            val_norm = -1.0 
        else:
            val_norm = minmax_scale(score, min_score, max_score)
            
        mapa_puntajes[uid] = val_norm
else:
    print(f"⚠️ NO SE ENCONTRÓ EL ARCHIVO DE PUNTAJES: {path_puntajes}")
    mapa_puntajes = {}

# ============================================================
# 🔹 3. VOCABULARIO DEL MODELO (2019+2020)
# ============================================================

if not os.path.exists(data_dir):
    print(f"❌ Error: No existe el directorio de tripletas: {data_dir}")
    sys.exit()

entities, relations = get_vocab_from_data_dir(data_dir)
print(f"✅ Vocabulario Dataset: {len(entities)} entidades, {len(relations)} relaciones.")

d = SimpleNamespace()
d.entities = entities
d.relations = relations
d.entity_idxs = {e: i for i, e in enumerate(entities)}
d.relation_idxs = {r: i for i, r in enumerate(relations)}

# ============================================================
# 🔹 4. ASIGNAR EMBEDDINGS (Dim 5)
# ============================================================

modelo = TuckER(d, edim, rdim, **kwargs)
alumnos_tripletas = sorted([h for h in heads_desde_tripletas(data_dir) if h in d.entity_idxs])

contador_inicializados = 0
contador_sin_datos = 0

print(f"Inyectando vectores de dimensión {edim}...")

with torch.no_grad():
    for alumno in alumnos_tripletas:
        # 1. Obtener Notas (Dim 4)
        if alumno in notas_s1_matrix.index:
            notas_vec = notas_s1_matrix.loc[alumno].values.astype(np.float32)
            contador_inicializados += 1
        else:
            # Si no está en el registro S1, se llena con -1 (desconocido)
            notas_vec = np.full(len(cursos_1er_semestre), -1.0, dtype=np.float32)
            contador_sin_datos += 1

        # 2. Obtener Puntaje (Dim 1)
        puntaje_val = mapa_puntajes.get(alumno, -1.0)
        
        # 3. Concatenar -> Vector final de 5 elementos
        vector_final = np.append(notas_vec, puntaje_val)
        
        # 4. Asignar al tensor
        idx = d.entity_idxs[alumno]
        modelo.E.weight[idx, :len(vector_final)] = torch.tensor(vector_final, dtype=torch.float32)

print(f"\n📊 Resumen:")
print(f"   Alumnos con historia (S1) : {contador_inicializados}")
print(f"   Alumnos sin historia (Cold): {contador_sin_datos}")

# ============================================================
# 🔹 5. GUARDAR
# ============================================================

embeddings_path = os.path.join(output_dir, "embeddings_inicializados_normalizados_5d_2cohortes.pt")
vocab_path = os.path.join(output_dir, "vocabulario_5d_2cohortes.json")

torch.save(modelo.E.weight.data, embeddings_path)
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump({"entities": d.entities, "relations": d.relations}, f, indent=2, ensure_ascii=False)

print(f"\n💾 Embeddings 5D guardados en: {embeddings_path}")
print(f"💾 Vocabulario guardado en: {vocab_path}")
print("✅ Listo para entrenar TuckER (2 Cohortes 5D).")

Cargando notas académicas (2019-1 y 2020-1)...
   Total alumnos con notas S1: 3242
Cargando y normalizando puntajes...
   Rango Puntaje detectado (2019+2020): [534.85, 935.2]
✅ Vocabulario Dataset: 1585 entidades, 4 relaciones.
Inyectando vectores de dimensión 5...

📊 Resumen:
   Alumnos con historia (S1) : 1577
   Alumnos sin historia (Cold): 0

💾 Embeddings 5D guardados en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_2cohortes\embeddings_inicializados_normalizados_5d_2cohortes.pt
💾 Vocabulario guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_2cohortes\vocabulario_5d_2cohortes.json
✅ Listo para entrenar TuckER (2 Cohortes 5D).


### .bat usado balanceado

In [ ]:
:: ================================================================
:: ENTRENAMIENTO MULTIPLE DE MODELOS TUCKER (5D - NOTAS + PUNTAJE)
:: ================================================================

@echo off
ECHO ===============================================================
ECHO            INICIANDO ENTRENAMIENTOS MULTIDIMENSIONALES (5D)
ECHO ===============================================================

:: Configuración general
set DATASET=dataset_2019_2020_fundamentales
:: ⚠️ RUTAS ACTUALIZADAS A TUS NUEVOS ARCHIVOS 5D
set EMB_INIT=notebooks/Experimento_warm_start/embeddings_5d_2cohortes/embeddings_inicializados_normalizados_5d_2cohortes.pt
set VOCAB_INIT=notebooks/Experimento_warm_start/embeddings_5d_2cohortes/vocabulario_5d_2cohortes.json

set BATCH=128
set LR=0.003
set PATIENCE=400
set EPOCHS=1000 

:: Lista de dimensiones de relación a entrenar
set RDIMS=1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16

:: Bucle principal
for %%R in (%RDIMS%) do (
    ECHO.
    ECHO ===============================================================
    ECHO Entrenando modelo con dimension de relaciones = %%R
    ECHO ===============================================================

    python main_original_warm_start_gemini_2_earlystopping_gemini.py ^
        --dataset %DATASET% ^
        --output_prefix Experimento_warm_start_rdim%%R_1000epochs_earlystopping_2019_2020_patience%PATIENCE%_5d ^
        --edim 5 ^
        --rdim %%R ^
        --num_iterations %EPOCHS% ^
        --batch_size %BATCH% ^
        --lr %LR% ^
        --init_embeddings %EMB_INIT% ^
        --init_vocab %VOCAB_INIT% ^
        --patience %PATIENCE%

    ECHO ---------------------------------------------------------------
    ECHO Modelo con rdim=%%R completado.
    ECHO ---------------------------------------------------------------
)

ECHO ===============================================================
ECHO TODOS LOS ENTRENAMIENTOS FINALIZADOS.
ECHO ===============================================================

pause

### Entrenar redes neuronales

In [1]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# -------------------------------------------------
# 1. Rutas del Modelo TuckER (2 COHORTES 5D)
# -------------------------------------------------
# Carpeta de tripletas (usada solo para vocabulario)
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"

# Carpeta de Resultados TuckER
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# ⚠️ AJUSTA ESTE PREFIJO SI ES DIFERENTE EN TU CARPETA RESULTS
# Debe coincidir con cómo se guardaron los modelos del .bat anterior
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience400_5d"

# -------------------------------------------------
# 2. Rutas de Datos y Salida
# -------------------------------------------------
BASE_DF_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RUTA_PUNTAJES   = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Carpeta de Salida para las Redes
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado"

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17))  # 1..16

# =========================
# FUNCIONES AUXILIARES
# =========================
def cargar_df_notas(path_csv):
    if not os.path.exists(path_csv): return pd.DataFrame()
    df = pd.read_csv(path_csv, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    for part in ['train.txt', 'valid.txt', 'test.txt']:
        path = os.path.join(data_dir, part)
        if not os.path.exists(path): continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1: entities.add(parts[0]); entities.add(parts[2])
    return sorted(list(entities))

def pick_state_dict(ckpt_loaded):
    if isinstance(ckpt_loaded, dict):
        if "model_state_dict" in ckpt_loaded: return ckpt_loaded["model_state_dict"]
        if "state_dict" in ckpt_loaded: return ckpt_loaded["state_dict"]
    return ckpt_loaded

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

# Normalizaciones (Deben ser idénticas a la fase de generación de embeddings)
def norm_nota(n):
    # (n - 4) / 3 -> [-1, 1]
    try:
        if pd.isna(n): return -1.0
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

def norm_puntaje(val, min_v, max_v):
    # MinMax -> [-1, 1]
    if pd.isna(val): return -1.0
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def construir_vectores_5d(df_sem1, df_sem2, df_ptje, cursos_primer, cursos_permitidos):
    # 1. Filtrar alumnos válidos (4 cursos S1 + Algo en S2)
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())

    if not alumnos_validos: return pd.DataFrame()

    # 2. Matriz de Notas
    df_notas = df_sem1[df_sem1['ID'].isin(alumnos_validos) & df_sem1['CURSO'].isin(cursos_primer)].copy()
    
    # Pivot
    pivot = df_notas.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    pivot = pivot.reindex(columns=cursos_primer)
    
    # Aplicar normalización a cada celda
    pivot = pivot.applymap(norm_nota)
    
    # 3. Agregar Puntaje
    # Calculamos min/max local de estos alumnos para consistencia
    ptjes_validos = df_ptje[df_ptje["ID"].isin(alumnos_validos)]["PUNTAJE_PONDERADO"].dropna()
    min_s = ptjes_validos.min() if not ptjes_validos.empty else 450.0
    max_s = ptjes_validos.max() if not ptjes_validos.empty else 850.0
    
    mapa_ptje = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    
    col_ptje = []
    for uid in pivot.index:
        raw_p = mapa_ptje.get(uid, np.nan)
        col_ptje.append(norm_puntaje(raw_p, min_s, max_s))
        
    pivot["PUNTAJE"] = col_ptje
    return pivot # DataFrame (N_alumnos, 5)

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, lr=1e-3, patience=100):
    X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)
    X_tr, X_val, Y_tr, Y_val         = train_test_split(X_train, Y_train, test_size=0.15, random_state=42)

    X_tr_t = torch.FloatTensor(X_tr); Y_tr_t = torch.FloatTensor(Y_tr)
    X_val_t= torch.FloatTensor(X_val);Y_val_t= torch.FloatTensor(Y_val)
    X_te_t = torch.FloatTensor(X_test);Y_te_t= torch.FloatTensor(Y_test)

    model = EmbeddingPredictor(X_tr.shape[1], Y_tr.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val, patience_counter = float('inf'), 0

    for epoch in range(1, max_epochs + 1):
        model.train(); optimizer.zero_grad()
        loss = criterion(model(X_tr_t), Y_tr_t)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad(): vloss = criterion(model(X_val_t), Y_val_t)

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    # Test final con el mejor
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    with torch.no_grad(): test_mse = criterion(model(X_te_t), Y_te_t).item()
    return test_mse

# =========================
# MAIN
# =========================
def main():
    if not os.path.exists(SAVE_BASE): os.makedirs(SAVE_BASE)
    print(f"📂 Guardando redes en: {SAVE_BASE}")
    print("=== Entrenando Redes Neuronales (2 Cohortes 5D: 2019-2020) ===")

    # 1. Vocabulario (para índices)
    vocab = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab)}
    
    # 2. Cargar Datos
    df_19_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_19_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    df_20_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))
    
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    # Limpiar puntaje
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')

    # 3. Construir Vectores 5D
    print("   Construyendo vectores 5D...")
    vecs_19 = construir_vectores_5d(df_19_1, df_19_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_20 = construir_vectores_5d(df_20_1, df_20_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    
    X_total_df = pd.concat([vecs_19, vecs_20], axis=0)
    # Eliminar duplicados si un alumno repitió año (usamos la primera aparición)
    X_total_df = X_total_df[~X_total_df.index.duplicated(keep='first')]
    
    print(f"-> Vectores X listos: {len(X_total_df)} alumnos (2019+2020). Dim={X_total_df.shape[1]}")

    # 4. Entrenar
    resumen = []
    for rdim in RDIMS:
        run_dir   = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        # Nombre específico para 2 cohortes 5d
        save_path = os.path.join(SAVE_BASE, f"best_predictor_rdim{rdim}_2cohortes_5d.pt")

        print(f"\n>> rdim={rdim}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER: {tucker_pt}")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            
            # Intersección
            alumnos_comunes = sorted(set(X_total_df.index).intersection(entity_idxs.keys()))
            if not alumnos_comunes:
                print("   ⚠️ Sin intersección de alumnos.")
                continue
                
            X_data = X_total_df.loc[alumnos_comunes].values.astype(np.float32)
            idxs   = [entity_idxs[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()
            
            mse = entrenar_predictor(X_data, Y_data, save_path)
            print(f"   ✅ Guardado. MSE: {mse:.6f}")
            resumen.append((rdim, mse))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")

    if resumen:
        pd.DataFrame(resumen, columns=["rdim", "mse"]).to_csv(os.path.join(SAVE_BASE, "resumen.csv"), index=False)
        print("\n✅ Proceso finalizado.")

if __name__ == "__main__":
    main()

📂 Guardando redes en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado
=== Entrenando Redes Neuronales (2 Cohortes 5D: 2019-2020) ===
   Construyendo vectores 5D...
-> Vectores X listos: 1571 alumnos (2019+2020). Dim=5

>> rdim=1
   ✅ Guardado. MSE: 0.022938

>> rdim=2
   ✅ Guardado. MSE: 0.023879

>> rdim=3
   ✅ Guardado. MSE: 0.015209

>> rdim=4
   ✅ Guardado. MSE: 0.013952

>> rdim=5
   ✅ Guardado. MSE: 0.015353

>> rdim=6
   ✅ Guardado. MSE: 0.014142

>> rdim=7
   ✅ Guardado. MSE: 0.010600

>> rdim=8
   ✅ Guardado. MSE: 0.009502

>> rdim=9
   ✅ Guardado. MSE: 0.014037

>> rdim=10
   ✅ Guardado. MSE: 0.011566

>> rdim=11
   ✅ Guardado. MSE: 0.011424

>> rdim=12
   ✅ Guardado. MSE: 0.013335

>> rdim=13
   ✅ Guardado. MSE: 0.007257

>> rdim=14
   ✅ Guardado. MSE: 0.009397

>> rdim=15
   ✅ Guardado. MSE: 0.011750

>> rdim=16
   ✅ Guardado. MSE: 0.009950

✅ Proceso finalizado.


### Entrenar red balanceada

In [5]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# -------------------------------------------------
# 1. Rutas (2 COHORTES 5D)
# -------------------------------------------------
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience400_5d"

BASE_DF_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RUTA_PUNTAJES   = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# ⚠️ CARPETA DE SALIDA (Le puse un sufijo _NN_BAL para diferenciar)
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado_NN_BAL"

CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17))

# =========================
# FUNCIONES AUXILIARES
# =========================
def cargar_df_notas(path_csv):
    if not os.path.exists(path_csv): return pd.DataFrame()
    df = pd.read_csv(path_csv, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    for part in ['train.txt', 'valid.txt', 'test.txt']:
        path = os.path.join(data_dir, part)
        if not os.path.exists(path): continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1: entities.add(parts[0]); entities.add(parts[2])
    return sorted(list(entities))

def cargar_E_weights(path_pt):
    sd = torch.load(path_pt, map_location=DEVICE)
    if "model_state_dict" in sd: sd = sd["model_state_dict"]
    elif "state_dict" in sd: sd = sd["state_dict"]
    return sd["E.weight"].detach().cpu()

# --- Normalización ---
def norm_nota(n):
    try:
        if pd.isna(n): return -1.0
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

def norm_puntaje(val, min_v, max_v):
    if pd.isna(val): return -1.0
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def construir_vectores_5d(df_sem1, df_sem2, df_ptje, cursos_primer, cursos_permitidos):
    # Filtro alumnos
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())
    if not alumnos_validos: return pd.DataFrame()

    # Notas
    df_notas = df_sem1[df_sem1['ID'].isin(alumnos_validos) & df_sem1['CURSO'].isin(cursos_primer)].copy()
    pivot = df_notas.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    pivot = pivot.reindex(columns=cursos_primer)
    pivot = pivot.applymap(norm_nota)
    
    # Puntaje
    ptjes_validos = df_ptje[df_ptje["ID"].isin(alumnos_validos)]["PUNTAJE_PONDERADO"].dropna()
    min_s = ptjes_validos.min() if not ptjes_validos.empty else 450.0
    max_s = ptjes_validos.max() if not ptjes_validos.empty else 850.0
    
    mapa_ptje = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    col_ptje = []
    for uid in pivot.index:
        raw_p = mapa_ptje.get(uid, np.nan)
        col_ptje.append(norm_puntaje(raw_p, min_s, max_s))
        
    pivot["PUNTAJE"] = col_ptje
    return pivot

# ============================================================
# ⚠️ FUNCIÓN DE BALANCEO (NUEVA)
# ============================================================
def balancear_xy(X_in, Y_in, factor=5):
    """
    Identifica alumnos con notas bajas (alguna nota < 4.0, es decir < 0.0 normalizado)
    y los duplica 'factor' veces en el set de entrenamiento.
    """
    # X_in es (N, 5). Las primeras 4 columnas son notas.
    # Si nota < 0.0 (normalizado), significa nota < 4.0.
    
    # Condición: ¿Tiene alguna nota roja en los 4 ramos fundamentales?
    # Usamos un umbral de -0.01 para evitar errores de flotante con el 0 exacto.
    mask_riesgo = (X_in[:, :4] < -0.01).any(axis=1)
    
    n_riesgo = mask_riesgo.sum()
    if n_riesgo == 0:
        return X_in, Y_in # Nada que balancear
        
    print(f"      ⚖️ Balanceando: {n_riesgo} alumnos en riesgo detectados.")
    print(f"      ⚖️ Multiplicando x{factor} (Oversampling)...")
    
    X_riesgo = X_in[mask_riesgo]
    Y_riesgo = Y_in[mask_riesgo]
    
    # Repetir
    X_extra = np.tile(X_riesgo, (factor, 1))
    Y_extra = np.tile(Y_riesgo, (factor, 1))
    
    # Concatenar
    X_bal = np.vstack([X_in, X_extra])
    Y_bal = np.vstack([Y_in, Y_extra])
    
    # Shuffle para mezclar
    perm = np.random.permutation(len(X_bal))
    return X_bal[perm], Y_bal[perm]

# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, lr=1e-3, patience=100):
    # Split Train/Test (sin balancear el test para que sea realista)
    X_train_raw, X_test, Y_train_raw, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)
    
    # ⚠️ BALANCEAR SOLO EL TRAINING SET
    # Aplicamos oversampling a los reprobados en el set de entrenamiento
    X_train_bal, Y_train_bal = balancear_xy(X_train_raw, Y_train_raw, factor=5)
    
    # Split Train/Val (del balanceado)
    X_tr, X_val, Y_tr, Y_val = train_test_split(X_train_bal, Y_train_bal, test_size=0.15, random_state=42)

    # A tensores
    X_tr_t = torch.FloatTensor(X_tr); Y_tr_t = torch.FloatTensor(Y_tr)
    X_val_t= torch.FloatTensor(X_val);Y_val_t= torch.FloatTensor(Y_val)
    X_te_t = torch.FloatTensor(X_test);Y_te_t= torch.FloatTensor(Y_test)

    model = EmbeddingPredictor(X_tr.shape[1], Y_tr.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val, patience_counter = float('inf'), 0

    for epoch in range(1, max_epochs + 1):
        model.train(); optimizer.zero_grad()
        loss = criterion(model(X_tr_t), Y_tr_t)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad(): vloss = criterion(model(X_val_t), Y_val_t)

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    with torch.no_grad(): test_mse = criterion(model(X_te_t), Y_te_t).item()
    return test_mse

# =========================
# MAIN
# =========================
def main():
    if not os.path.exists(SAVE_BASE): os.makedirs(SAVE_BASE)
    print(f"📂 Guardando redes BALANCEADAS en: {SAVE_BASE}")
    print("=== Entrenando Redes 2 Cohortes 5D (Con Oversampling de Riesgo) ===")

    # 1. Vocabulario
    vocab = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab)}
    
    # 2. Cargar Datos
    df_19_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_19_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    df_20_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))
    
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')

    # 3. Construir Vectores
    print("   Construyendo vectores...")
    vecs_19 = construir_vectores_5d(df_19_1, df_19_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_20 = construir_vectores_5d(df_20_1, df_20_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    
    X_total_df = pd.concat([vecs_19, vecs_20], axis=0)
    X_total_df = X_total_df[~X_total_df.index.duplicated(keep='first')]
    
    print(f"-> Vectores X listos: {len(X_total_df)} alumnos.")

    # 4. Entrenar
    resumen = []
    for rdim in RDIMS:
        run_dir   = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_rdim{rdim}_2cohortes_5d.pt")

        print(f"\n>> rdim={rdim}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER: {tucker_pt}")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            alumnos_comunes = sorted(set(X_total_df.index).intersection(entity_idxs.keys()))
            if not alumnos_comunes: continue
                
            X_data = X_total_df.loc[alumnos_comunes].values.astype(np.float32)
            idxs   = [entity_idxs[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()
            
            # Aquí ocurre la magia del balanceo (dentro de entrenar_predictor)
            mse = entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, patience=50)
            print(f"   ✅ Guardado. Test MSE: {mse:.6f}")
            resumen.append((rdim, mse))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")

    if resumen:
        pd.DataFrame(resumen, columns=["rdim", "mse"]).to_csv(os.path.join(SAVE_BASE, "resumen.csv"), index=False)
        print("\n✅ Proceso finalizado.")

if __name__ == "__main__":
    main()

📂 Guardando redes BALANCEADAS en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado_NN_BAL
=== Entrenando Redes 2 Cohortes 5D (Con Oversampling de Riesgo) ===
   Construyendo vectores...
-> Vectores X listos: 1571 alumnos.

>> rdim=1
      ⚖️ Balanceando: 799 alumnos en riesgo detectados.
      ⚖️ Multiplicando x5 (Oversampling)...
   ✅ Guardado. Test MSE: 0.027125

>> rdim=2
      ⚖️ Balanceando: 799 alumnos en riesgo detectados.
      ⚖️ Multiplicando x5 (Oversampling)...
   ✅ Guardado. Test MSE: 0.027042

>> rdim=3
      ⚖️ Balanceando: 799 alumnos en riesgo detectados.
      ⚖️ Multiplicando x5 (Oversampling)...
   ✅ Guardado. Test MSE: 0.015997

>> rdim=4
      ⚖️ Balanceando: 799 alumnos en riesgo detectados.
      ⚖️ Multiplicando x5 (Oversampling)...
   ✅ Guardado. Test MSE: 0.017882

>> rdim=5
      ⚖️ Balanceando: 799 alumnos en riesgo detectados.
      ⚖️ Multiplicando x5 (Oversampling)...
   ✅ Guardado. Test MSE: 0.015087

>> rdim=6
      

### Probar modelo balanceado, sin red aumentada

In [3]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# =====================================
# 1. CONFIGURACIÓN
# =====================================

# Directorio del Dataset (Vocabulario)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\\"

# DATOS DE PRUEBA (Cohorte 2021, ya que entrenaste con 19-20)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_INPUT = os.path.join(BASE_PATH, "df_20211.csv")  # S1
CSV_TARGET = os.path.join(BASE_PATH, "df_20212.csv") # S2
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Rutas de Modelos
RESULTS_BASE   = r"C:\Users\56946\TuckER\results"
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado"

# Nombres de archivos
RUN_PREFIX         = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience400_5d"
PREDICTOR_FILE_FMT = "best_predictor_rdim{rdim}_2cohortes_5d.pt"

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"
RDIMS = list(range(1, 17))

# =========================
# UTILIDADES
# =========================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data

def build_vocab(data_dir, reverse=True):
    if not data_dir.endswith(os.sep): data_dir += os.sep
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    E = sd["E.weight"].to(device)
    R = sd["R.weight"].to(device)
    W = sd["W"].to(device)
    return E, R, W

def pick_forward_rel_idx(vocab, name):
    if name in vocab.relation_idxs: return vocab.relation_idxs[name]
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands: cands = [r for r in vocab.relations if re.search(name, r, re.I)]
    if not cands: raise KeyError(f"No encontré relación '{name}' en el vocab.")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    raise ValueError(f"Layout W no reconocido: {tuple(W.shape)}")

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    model.load_state_dict(pick_state_dict(torch.load(path, map_location=DEVICE)), strict=True)
    model.to(DEVICE).eval()
    return model

# ⚠️ CONSTRUCTOR DE VECTOR 5D (Igual al entrenamiento)
def construir_vector_5d(csv_path, alumno_id, cursos_primer, puntaje_val, min_s, max_s):
    # 1. Notas
    df = pd.read_csv(csv_path, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    
    sub = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    def clean_n(x):
        try: return float(str(x).replace(",", "."))
        except: return 0.0 # Si no hay nota en input, es 0.0 (o 1.0, usaremos 1.0 si vacío para ser consistente con riesgo)
    
    # Si está vacío el input S1, asumimos riesgo (1.0)
    if sub.empty:
        vec_notas = np.full(len(cursos_primer), 1.0, dtype=np.float32)
    else:
        sub['NOTA'] = sub['NOTA'].apply(clean_n)
        series = sub.drop_duplicates(subset=['CURSO']).set_index('CURSO')['NOTA']
        series = series.reindex(cursos_primer, fill_value=1.0) # Rellenar faltantes con 1.0
        vec_notas = series.values.astype(np.float32)

    # Normalizar notas (n-4)/3
    vec_notas_norm = (vec_notas - 4.0) / 3.0
    
    # 2. Puntaje
    if max_s == min_s: p_norm = 0.0
    else: p_norm = 2 * (puntaje_val - min_s) / (max_s - min_s) - 1
    if p_norm < -1.0: p_norm = -1.0 # Caso sin puntaje (-1 original) se mantiene bajo

    # 3. Concatenar
    final = np.append(vec_notas_norm, p_norm)
    return torch.tensor(final, dtype=torch.float32).view(1, -1)

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab, mapa_puntajes, min_s, max_s):
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        ridx_apr = pick_forward_rel_idx(vocab, "aprueba")
        try: ridx_repr = pick_forward_rel_idx(vocab, "reprueba"); have_repr = True
        except: have_repr = False

        predictor = load_predictor(predictor_ckpt, 5, d_e) # 5 DIMENSIONES
    except Exception as e:
        print(f"❌ Error cargando {tag}: {e}")
        return None

    # Cargar Datos Test
    df_in = pd.read_csv(CSV_INPUT, sep=';')
    df_out = pd.read_csv(CSV_TARGET, sep=';')
    for df in (df_in, df_out):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    ok = (df_in.groupby("ID")["CURSO"].apply(set).apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    alumnos_validos = ok[ok].index.tolist()

    df_eval = df_out[(df_out['ID'].isin(alumnos_validos)) & (df_out['CURSO'].isin(CURSOS_EVAL))].copy()

    # Target Label
    def clean_target(x):
        try:
            s = str(x).strip()
            if not s or s.lower() == 'nan': return 1.0 # Reprueba
            return float(s.replace(",", "."))
        except: return 1.0
    
    df_eval['VAL_NOTA'] = df_eval['NOTA'].apply(clean_target)
    df_eval['APROB'] = (df_eval['VAL_NOTA'] >= 4.0).astype(int)
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())].copy()

    if df_eval.empty: return None

    ehat_cache = {}
    rows = []
    y_true, y_pred = [], []

    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_repr = contract_M(W, R[ridx_repr], d_e) if have_repr else None

        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            
            if aid not in ehat_cache:
                ptje = mapa_puntajes.get(aid, -1.0) # -1 si no existe
                x = construir_vector_5d(CSV_INPUT, aid, CURSOS_PRIMER, ptje, min_s, max_s).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0).cpu()
            
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]].cpu()

            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item() if have_repr else 1.0 - s_apr

            # Clasificación
            real_repro = 1 if r['APROB'] == 0 else 0
            pred_repro = 1 if s_repr > s_apr else 0
            
            y_true.append(real_repro)
            y_pred.append(pred_repro)

            # Ranking Hit@1
            hit = 1.0 if ((r['APROB']==1 and s_apr>=s_repr) or (r['APROB']==0 and s_repr>=s_apr)) else 0.0
            rows.append({"hit@1": hit})

    # Métricas
    df_res = pd.DataFrame(rows)
    acc = df_res["hit@1"].mean()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    n_reales = fn + tp
    n_predichos = tp + fp

    # IMPRESIÓN CON EL FORMATO SOLICITADO
    print(f"\n============================================================")
    print(f" 📊 RESULTADOS CRITERIO ESTÁNDAR ({tag})")
    print(f"============================================================")
    print(f"Total Evaluaciones              : {len(df_eval)}")
    print(f"Accuracy Global (Hit@1)         : {acc:.4f}")
    print(f"------------------------------------------------------------")
    print(f"Total REALES Reprobados         : {n_reales}")
    print(f"Total PREDICHOS Reprobados      : {n_predichos}")
    print(f"------------------------------------------------------------")
    print(f"✅ RECALL (Sensibilidad) Reprueba: {rec:.4f}")
    print(f"   (Detectamos {tp} de {n_reales} reprobados)")
    print(f"------------------------------------------------------------")
    print(f"🎯 PRECISION Reprueba           : {prec:.4f}")
    print(f"   (De los {n_predichos} que dijimos que reprobaban, {tp} eran reales)")
    print(f"============================================================")
    print(f"\nDesglose:")
    print(f"Correctos Reprobados : {tp}")
    print(f"Perdidos (Peligrosos): {fn} (Predijo Aprueba, era Reprueba)")
    print(f"Falsas Alarmas       : {fp} (Predijo Reprueba, era Aprueba)")

    return {"tag": tag, "Hits@1": acc, "Precision_Rep": prec, "Recall_Rep": rec, "N": len(df_res)}

def main():
    if not os.path.exists(DATA_DIR):
        print(f"❌ Error Data Dir: {DATA_DIR}")
        return

    vocab = build_vocab(DATA_DIR, reverse=True)
    
    # Cargar Puntajes Globales para normalización consistente
    print("--- Cargando Puntajes ---")
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
    scores = df_ptje["PUNTAJE_PONDERADO"].dropna()
    min_s, max_s = scores.min(), scores.max()
    mapa_puntajes = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    print(f"   Rango Puntaje: [{min_s}, {max_s}]")

    res_list = []
    print(f"📂 Evaluando 2 Cohortes 5D en Test 2021...")

    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt) or not os.path.exists(predictor_ckpt):
            print(f"⏩ Faltan archivos para {tag}")
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab, mapa_puntajes, min_s, max_s)
        if res: res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list).sort_values("tag")
        out = os.path.join(PREDICTOR_BASE, "resumen_evaluacion_2cohortes_5d_final.csv")
        df_sum.to_csv(out, index=False)
        print("\n================ RESUMEN 2 COHORTES 5D ================")
        print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        print(f"\n💾 Guardado en: {out}")

if __name__ == "__main__":
    main()

--- Cargando Puntajes ---
   Rango Puntaje: [534.85, 970.7]
📂 Evaluando 2 Cohortes 5D en Test 2021...

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim1)
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.2326
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 1963
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.3391
   (Detectamos 79 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.0402
   (De los 1963 que dijimos que reprobaban, 79 eran reales)

Desglose:
Correctos Reprobados : 79
Perdidos (Peligrosos): 154 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 1884 (Predijo Reprueba, era Aprueba)

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim2)
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.4386
------------------------------------------------------------
T


 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim11)
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.6236
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 657
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.0730
   (Detectamos 17 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.0259
   (De los 657 que dijimos que reprobaban, 17 eran reales)

Desglose:
Correctos Reprobados : 17
Perdidos (Peligrosos): 216 (Predijo Aprueba, era Reprueba)
Falsas Alarmas       : 640 (Predijo Reprueba, era Aprueba)

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim12)
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.8296
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 0
------------------------------

In [6]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# =====================================
# 1. CONFIGURACIÓN
# =====================================

# Vocabulario (Dataset 2 cohortes original)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\\"

# DATOS DE PRUEBA: COHORTE 2021 (Futuro respecto a 2019-2020)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_INPUT = os.path.join(BASE_PATH, "df_20211.csv")  # Input S1
CSV_TARGET = os.path.join(BASE_PATH, "df_20212.csv") # Target S2
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Rutas de Modelos
RESULTS_BASE   = r"C:\Users\56946\TuckER\results"

# ⚠️ CARPETA DONDE GUARDASTE LAS REDES CON OVERSAMPLING
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_5d_balanceado_NN_BAL"

# Patrones
RUN_PREFIX         = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience400_5d"
PREDICTOR_FILE_FMT = "best_predictor_rdim{rdim}_2cohortes_5d.pt"

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"
RDIMS = list(range(1, 17))

# =========================
# UTILIDADES
# =========================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data

def build_vocab(data_dir, reverse=True):
    if not data_dir.endswith(os.sep): data_dir += os.sep
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    E = sd["E.weight"].to(device)
    R = sd["R.weight"].to(device)
    W = sd["W"].to(device)
    return E, R, W

def pick_forward_rel_idx(vocab, name):
    if name in vocab.relation_idxs: return vocab.relation_idxs[name]
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands: cands = [r for r in vocab.relations if re.search(name, r, re.I)]
    if not cands: raise KeyError(f"No encontré relación '{name}' en el vocab.")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    raise ValueError(f"Layout W no reconocido: {tuple(W.shape)}")

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    model.load_state_dict(pick_state_dict(torch.load(path, map_location=DEVICE)), strict=True)
    model.to(DEVICE).eval()
    return model

# ⚠️ CONSTRUCTOR 5D ROBUSTO (Notas + Puntaje)
def construir_vector_5d(csv_path, alumno_id, cursos_primer, puntaje_val, min_s, max_s):
    df = pd.read_csv(csv_path, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    
    sub = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    def clean_n(x):
        try: return float(str(x).replace(",", "."))
        except: return 0.0 
    
    if sub.empty:
        # Si no hay notas, asumimos riesgo máximo (1.0)
        vec_notas = np.full(len(cursos_primer), 1.0, dtype=np.float32)
    else:
        sub['NOTA'] = sub['NOTA'].apply(clean_n)
        series = sub.drop_duplicates(subset=['CURSO']).set_index('CURSO')['NOTA']
        # Si falta algún ramo, rellenamos con 1.0 (Riesgo/Reprobado)
        series = series.reindex(cursos_primer, fill_value=1.0)
        vec_notas = series.values.astype(np.float32)

    # Normalizar notas (n-4)/3 -> [-1, 1]
    vec_notas_norm = (vec_notas - 4.0) / 3.0
    
    # 2. Puntaje Normalizado [-1, 1]
    if pd.isna(puntaje_val):
        p_norm = -1.0
    else:
        if max_s == min_s: p_norm = 0.0
        else: p_norm = 2 * (puntaje_val - min_s) / (max_s - min_s) - 1
        
    if p_norm < -1.0: p_norm = -1.0

    # 3. Concatenar
    final = np.append(vec_notas_norm, p_norm)
    return torch.tensor(final, dtype=torch.float32).view(1, -1)

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab, mapa_puntajes, min_s, max_s):
    print(f"\n--- Evaluando {tag} ---")
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        ridx_apr = pick_forward_rel_idx(vocab, "aprueba")
        try: ridx_repr = pick_forward_rel_idx(vocab, "reprueba"); have_repr = True
        except: have_repr = False

        predictor = load_predictor(predictor_ckpt, 5, d_e) # 5D
    except Exception as e:
        print(f"❌ Error cargando: {e}")
        return None

    # Cargar Datos Test (2021)
    df_in = pd.read_csv(CSV_INPUT, sep=';')
    df_out = pd.read_csv(CSV_TARGET, sep=';')
    for df in (df_in, df_out):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    ok = (df_in.groupby("ID")["CURSO"].apply(set).apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    alumnos_validos = ok[ok].index.tolist()

    df_eval = df_out[(df_out['ID'].isin(alumnos_validos)) & (df_out['CURSO'].isin(CURSOS_EVAL))].copy()

    # Target Label: Vacío o < 4.0 es Reprobado
    def clean_target(x):
        try:
            s = str(x).strip()
            if not s or s.lower() == 'nan': return 1.0 # Reprueba
            return float(s.replace(",", "."))
        except: return 1.0
    
    df_eval['VAL_NOTA'] = df_eval['NOTA'].apply(clean_target)
    df_eval['APROB'] = (df_eval['VAL_NOTA'] >= 4.0).astype(int)
    
    # Solo cursos conocidos
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())].copy()

    if df_eval.empty: return None

    ehat_cache = {}
    rows = []
    y_true, y_pred = [], []

    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_repr = contract_M(W, R[ridx_repr], d_e) if have_repr else None

        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            
            if aid not in ehat_cache:
                ptje = mapa_puntajes.get(aid, -1.0)
                x = construir_vector_5d(CSV_INPUT, aid, CURSOS_PRIMER, ptje, min_s, max_s).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0).cpu()
            
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]].cpu()

            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item() if have_repr else 1.0 - s_apr

            real_repro = 1 if r['APROB'] == 0 else 0
            pred_repro = 1 if s_repr > s_apr else 0
            
            y_true.append(real_repro)
            y_pred.append(pred_repro)

            hit = 1.0 if ((r['APROB']==1 and s_apr>=s_repr) or (r['APROB']==0 and s_repr>=s_apr)) else 0.0
            rows.append({"hit@1": hit})

    # Métricas
    df_res = pd.DataFrame(rows)
    acc = df_res["hit@1"].mean()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    
    n_reales = fn + tp
    n_predichos = tp + fp

    print(f"\n============================================================")
    print(f" 📊 RESULTADOS CRITERIO ESTÁNDAR ({tag}) - NN BALANCEADA")
    print(f"============================================================")
    print(f"Total Evaluaciones              : {len(df_eval)}")
    print(f"Accuracy Global (Hit@1)         : {acc:.4f}")
    print(f"------------------------------------------------------------")
    print(f"Total REALES Reprobados         : {n_reales}")
    print(f"Total PREDICHOS Reprobados      : {n_predichos}")
    print(f"------------------------------------------------------------")
    print(f"✅ RECALL (Sensibilidad) Reprueba: {rec:.4f}")
    print(f"   (Detectamos {tp} de {n_reales} reprobados)")
    print(f"------------------------------------------------------------")
    print(f"🎯 PRECISION Reprueba           : {prec:.4f}")
    print(f"   (De los {n_predichos} que dijimos que reprobaban, {tp} eran reales)")
    print(f"============================================================")
    print(f"Desglose: [TP={tp}] [FN={fn}] [FP={fp}]")

    return {"tag": tag, "Hits@1": acc, "Precision_Rep": prec, "Recall_Rep": rec, "N": len(df_res)}

def main():
    if not os.path.exists(DATA_DIR): return

    vocab = build_vocab(DATA_DIR, reverse=True)
    
    print("--- Cargando Puntajes Globales ---")
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
    scores = df_ptje["PUNTAJE_PONDERADO"].dropna()
    min_s, max_s = scores.min(), scores.max()
    mapa_puntajes = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    
    res_list = []
    print(f"📂 Evaluando 2 Cohortes 5D (Test 2021) - Redes Balanceadas...")

    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt) or not os.path.exists(predictor_ckpt):
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab, mapa_puntajes, min_s, max_s)
        if res: res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list).sort_values("tag")
        out = os.path.join(PREDICTOR_BASE, "resumen_evaluacion_2cohortes_5d_BAL_final.csv")
        df_sum.to_csv(out, index=False)
        print("\n================ RESUMEN 2 COHORTES 5D (BAL) ================")
        print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        print(f"\n💾 Guardado en: {out}")

if __name__ == "__main__":
    main()

--- Cargando Puntajes Globales ---
📂 Evaluando 2 Cohortes 5D (Test 2021) - Redes Balanceadas...

--- Evaluando rdim1 ---

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim1) - NN BALANCEADA
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.2326
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 1963
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.3391
   (Detectamos 79 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.0402
   (De los 1963 que dijimos que reprobaban, 79 eran reales)
Desglose: [TP=79] [FN=154] [FP=1884]

--- Evaluando rdim2 ---

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim2) - NN BALANCEADA
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.4386
------------------------------------------------------------
Total REALES Reprobados         : 233
Total 


 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim11) - NN BALANCEADA
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.6236
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 657
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprueba: 0.0730
   (Detectamos 17 de 233 reprobados)
------------------------------------------------------------
🎯 PRECISION Reprueba           : 0.0259
   (De los 657 que dijimos que reprobaban, 17 eran reales)
Desglose: [TP=17] [FN=216] [FP=640]

--- Evaluando rdim12 ---

 📊 RESULTADOS CRITERIO ESTÁNDAR (rdim12) - NN BALANCEADA
Total Evaluaciones              : 3023
Accuracy Global (Hit@1)         : 0.8296
------------------------------------------------------------
Total REALES Reprobados         : 233
Total PREDICHOS Reprobados      : 0
------------------------------------------------------------
✅ RECALL (Sensibilidad) Reprue